# Code 14.3.1: The DPO loss on one pair

The first half evaluates the implicit rewards on the numerical pair from this section. The second half optimizes a three-way softmax (chosen, rejected, other) from a tie, for five steps. The reference is the initial policy, so the implicit margin starts at zero.


In [ ]:
import math
import torch
import torch.nn.functional as F

beta = 0.1
logp_w, logp_l = -1.20, -0.80
ref_w, ref_l = -1.00, -1.10
r_w = beta * (logp_w - ref_w)
r_l = beta * (logp_l - ref_l)
margin = r_w - r_l
loss = -F.logsigmoid(torch.tensor(margin))
print(f"implicit rewards {r_w:.3f} {r_l:.3f}  margin {margin:.3f}  "
      f"loss {loss:.3f}  P(prefer chosen) {torch.sigmoid(torch.tensor(margin)):.3f}")

logits = torch.tensor([0.2, 0.5, 0.0], requires_grad=True)   # chosen, rejected, other
ref = F.log_softmax(logits.detach(), dim=0)
opt = torch.optim.SGD([logits], lr=0.5)
for step in range(5):
    logp = F.log_softmax(logits, dim=0)
    h = (logp[0] - ref[0]) - (logp[1] - ref[1])
    loss = -F.logsigmoid(beta * h)
    opt.zero_grad()
    loss.backward()
    opt.step()
    with torch.no_grad():
        pi = F.softmax(logits, dim=0)
        h = (F.log_softmax(logits, dim=0)[0] - ref[0]) - (F.log_softmax(logits, dim=0)[1] - ref[1])
        print(f"step {step}: loss {loss.item():.4f} -> pi {[round(p, 4) for p in pi.tolist()]}  "
              f"margin {beta * h.item():.4f}")


**Expected output**

Output:

```text
implicit rewards -0.020 0.030  margin -0.050  loss 0.718  P(prefer chosen) 0.488
step 0: loss 0.6931 -> pi [0.3244, 0.4165, 0.259]  margin 0.0050
step 1: loss 0.6907 -> pi [0.3333, 0.4071, 0.2596]  margin 0.0100
step 2: loss 0.6882 -> pi [0.3422, 0.3978, 0.26]  margin 0.0150
step 3: loss 0.6857 -> pi [0.3512, 0.3885, 0.2603]  margin 0.0199
step 4: loss 0.6832 -> pi [0.3603, 0.3792, 0.2605]  margin 0.0249
```

The loss printed on each line is the loss *before* that step; the probabilities are *after* it. The chosen response gains what the rejected response loses. The third probability stays near a quarter.
